## PBL(2) Project 를 위한 실습 (Exercise problem) — DNA 서열 분류

1. https://agtech.pythonanywhere.com/ 에 접속하여 회원가입해 주세요. (비밀번호는 단순하게 만드는 것을 권장. 예: 1234)
2. `username` 에 이메일 형식의 아이디를 기입해 주세요.
3. `password` 에 비밀번호를 기입해 주세요.

In [ ]:
project = "dnasequence"  # 수정하지 마세요
username = ""  # 회원가입 시 사용한 이메일아이디 (예시. abc@hello.com)
password = ""  # 비밀번호

리더보드 제출을 위한 기본 설정: 아래 코드를 실행해주세요.

In [ ]:
import os
import urllib.request

if not os.path.exists("competition.py"):
    url = "https://raw.githubusercontent.com/agtechresearch/LectureMLbasic/refs/heads/main/competition/competition.py"
    filename = "competition.py"
    urllib.request.urlretrieve(url, filename)

아래 코드를 실행하여 데이터를 다운로드 받습니다: 3개의 csv 파일이 data 폴더에 다운로드됨

 * dataset.csv: 과거 인간 게놈에서 수집된 DNA 서열과 조절 영역 라벨(0/1/2) → 학습에 사용할 데이터셋 (50,000행)
 * problem.csv: 현재 분류 대상이 되는 10,000건의 DNA 서열 → ML 모델에 의하여 예측(0/1/2)을 수행하여야 할 데이터셋
 * submission.csv: 리더보드 서버 제출을 위한 파일 형식

In [ ]:
import competition

# 파일 다운로드 (GitHub raw)
competition.download_competition_files(
    "https://raw.githubusercontent.com/agtechresearch/LectureMLbasic/main/dnasequence/bundle.zip",
    use_competition_url=False,
)

----------

### 아래는 k-mer 빈도 + 로지스틱 회귀를 사용하여 DNA 조절 영역 분류 모델을 만들고, 코랩환경에서 결과를 리더보드에 제출하는 간단한 샘플 코드입니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 경고 무시
warnings.filterwarnings("ignore")

# Data 경로 설정
DATA_DIR = "data"

In [ ]:
# 학습에 사용할 과거 DNA 서열 data set 로드 (dataset.csv)
dataset = pd.read_csv(os.path.join(DATA_DIR, "dataset.csv"))

# problem set 로드 (problem.csv)
problemset = pd.read_csv(os.path.join(DATA_DIR, "problem.csv"))

In [ ]:
dataset  # 학습에 사용할 과거 DNA 서열 데이터셋 확인: 50,000건

### **<데이터 구성>**

* `seq` : DNA 서열 문자열. 알파벳은 `A / C / G / T / N` 5종으로 구성되며, 길이는 가변.
  * `A`, `C`, `G`, `T` : 4종의 염기(base)
  * `N` : unknown base (서열 중 약 5%)
* `label` : 조절 영역 클래스 (0 / 1 / 2). 원본 클래스명은 `enhancer` / `promoter` / `OCR`.

**문제 목표**: `problem.csv`의 10,000건 DNA 서열에 대해 `label` (0/1/2)을 예측하여 제출.

In [ ]:
# problem set 확인: 10,000건의 문제 데이터셋 (label을 예측해야 함)
problemset

In [ ]:
# 클래스 분포 확인
print("클래스 분포:")
print(dataset["label"].value_counts().sort_index())

# 서열 길이 분포 확인
print("\n서열 길이 통계:")
print(dataset["seq"].str.len().describe())

## 데이터 전처리 및 모델 학습

DNA 서열은 문자열이라 모델에 직접 넣을 수 없습니다. 가장 단순한 방법으로, 서열 안의 **각 염기 문자(A/C/G/T/N) 빈도**를 세서 5개의 숫자 피처로 변환합니다.

In [ ]:
# 서열 문자열을 A/C/G/T/N 문자 빈도(5개 피처)로 변환
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer="char", lowercase=False)
X_all = vectorizer.fit_transform(dataset["seq"])
X_problem = vectorizer.transform(problemset["seq"])

Y = dataset["label"].values

print("학습 데이터 shape:", X_all.shape)
print("문제 데이터 shape:", X_problem.shape)

In [ ]:
# 모델 학습을 위해 학습 데이터를 80%의 학습 데이터(train)와 20%의 검증 데이터(test)로 나눔
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X_all, Y, test_size=0.2, stratify=Y, random_state=42
)

In [ ]:
# 로지스틱 회귀 모델을 사용하여 학습
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(x_train, y_train)

In [ ]:
# train 데이터와 test 데이터에 대한 예측값을 구하고 Accuracy 값을 계산
from sklearn.metrics import accuracy_score

train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

print("Train Accuracy :", accuracy_score(y_train, train_pred))
print("Test Accuracy  :", accuracy_score(y_test, test_pred))

Accuracy: 정확도 (예측이 정답과 일치한 비율)

## Problem set 문제에 대한 클래스 예측 및 리더보드 결과 제출

- 아래 제출 프로세스가 느리다고 중지 후 다시 코드를 여러차례 재실행하는 경우 패널티가 발생할 수 있습니다. (제출 과정에서 제출 횟수 이슈 발생 가능: 하루 최대 100회 까지 가능)
- 제출에 성공할 경우, "제출에 성공하였습니다"의 메세지와 함께 제출 결과 Accuracy 가 화면에 출력됩니다.
- 제출결과는 또한 [대회 페이지(리더보드 서버)](https://agtech.pythonanywhere.com/competitions/dnasequence/)의 `리더보드` 와 `제출` 탭에서 확인할 수 있습니다.

In [ ]:
# 문제 데이터(problem data)에 대한 예측값을 구함
problem_pred = model.predict(X_problem)

In [ ]:
# 리더보드 서버 제출을 위한 파일 생성
submission = pd.read_csv(os.path.join(DATA_DIR, "submission.csv"))
submission["label"] = problem_pred

# 예측 결과 화면에 출력 후 제출
display(submission)
competition.submit(project, username, password, submission)